# PostGIS Basics

## Overview
PostGIS is a PostgreSQL extension that adds geometry types and spatial functions. These notebooks use GeoPandas + Shapely to run all examples live, with complete PostGIS SQL shown throughout.

**PostGIS setup:**
```sql
CREATE EXTENSION postgis;   -- run once per database
SELECT postgis_full_version();

-- Table with geometry column:
CREATE TABLE monitoring_sites (
    site_id   TEXT PRIMARY KEY,
    site_name TEXT NOT NULL,
    province  TEXT,
    geom      geometry(Point, 4326)   -- Point geometry, SRID 4326
);
CREATE INDEX idx_sites_geom ON monitoring_sites USING GIST (geom);
```

**Dataset:** 8 ecological monitoring sites, 3 protected areas, 3 watersheds, 10 species observations across Atlantic Canada.

---

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, Polygon, LineString, MultiPolygon
import pyproj

# ── Ecological dataset: Atlantic Canada ───────────────────────────
# Monitoring sites, species observations, protected areas, watersheds

# Monitoring sites (water quality / biodiversity)
sites_data = {
    'site_id':   ['S01','S02','S03','S04','S05','S06','S07','S08'],
    'site_name': ['Petitcodiac River','Miramichi Lake','Fundy Shore',
                  'Tantramar Marsh','Kouchibouguac','Restigouche River',
                  'Saint John River','Annapolis Valley'],
    'province':  ['NB','NB','NB','NB','NB','NB','NB','NS'],
    'elevation_m':[12, 84, 5, 3, 8, 45, 18, 62],
    'geometry':  [
        Point(-64.96, 46.07), Point(-66.18, 46.85), Point(-65.07, 45.55),
        Point(-64.40, 45.90), Point(-64.95, 46.82), Point(-67.60, 47.70),
        Point(-66.64, 45.95), Point(-65.12, 45.05),
    ],
}
sites = gpd.GeoDataFrame(sites_data, crs='EPSG:4326')

# Protected areas (simplified polygons)
prot_data = {
    'pa_id':    ['PA01','PA02','PA03'],
    'name':     ['Fundy National Park','Kouchibouguac NP','Kejimkujik NP'],
    'type':     ['National Park','National Park','National Park'],
    'area_km2': [207, 239, 404],
    'geometry': [
        Polygon([(-65.4,45.4),(-64.6,45.4),(-64.6,45.7),(-65.4,45.7)]),
        Polygon([(-64.9,46.6),(-64.7,46.6),(-64.7,47.0),(-64.9,47.0)]),
        Polygon([(-65.4,44.5),(-64.8,44.5),(-64.8,44.9),(-65.4,44.9)]),
    ],
}
protected = gpd.GeoDataFrame(prot_data, crs='EPSG:4326')

# Watersheds
ws_data = {
    'ws_id':   ['WS01','WS02','WS03'],
    'name':    ['Saint John Watershed','Miramichi Watershed','Petitcodiac Watershed'],
    'area_km2':[55200, 13800, 3100],
    'geometry':[
        Polygon([(-68.0,46.8),(-66.0,46.8),(-66.0,45.0),(-68.0,45.0)]),
        Polygon([(-66.5,47.5),(-65.5,47.5),(-65.5,46.5),(-66.5,46.5)]),
        Polygon([(-65.2,46.3),(-64.5,46.3),(-64.5,45.7),(-65.2,45.7)]),
    ],
}
watersheds = gpd.GeoDataFrame(ws_data, crs='EPSG:4326')

# Species observations
obs_data = {
    'obs_id':   range(1, 11),
    'species':  ['Atlantic Salmon','Atlantic Salmon','Blanding\'s Turtle',
                 'Piping Plover','Piping Plover','Blanding\'s Turtle',
                 'Atlantic Salmon','Little Brown Bat','Little Brown Bat','Piping Plover'],
    'year':     [2022,2023,2022,2023,2022,2023,2021,2023,2022,2021],
    'count':    [12, 8, 3, 1, 2, 1, 15, 47, 52, 3],
    'geometry': [
        Point(-64.96,46.07), Point(-67.60,47.70), Point(-64.40,45.90),
        Point(-65.07,45.55), Point(-64.95,46.82), Point(-66.18,46.85),
        Point(-66.64,45.95), Point(-66.18,46.85), Point(-64.96,46.07),
        Point(-64.40,45.90),
    ],
}
observations = gpd.GeoDataFrame(obs_data, crs='EPSG:4326')

print("Ecological dataset loaded:")
for name, gdf in [('sites',sites),('protected_areas',protected),
                  ('watersheds',watersheds),('observations',observations)]:
    print(f"  {name:<18s}: {len(gdf)} features, CRS={gdf.crs.to_epsg()}")


---
## Distance, area, and basic measurements

In [ ]:
print("=== Basic PostGIS functions: distance, area, centroid ===")
print()

# Distance between sites (in degrees, EPSG:4326)
from shapely.ops import transform as shp_transform
import pyproj, functools

# Reproject to UTM 20N for accurate distances
proj_4326_to_utm = pyproj.Transformer.from_crs("EPSG:4326","EPSG:32620", always_xy=True).transform

def to_utm(geom):
    return shp_transform(proj_4326_to_utm, geom)

sites_utm = sites.to_crs("EPSG:32620")
obs_utm   = observations.to_crs("EPSG:32620")

print("Distances from Petitcodiac River (S01) to all other sites (metres):")
s01 = sites_utm[sites_utm.site_id=='S01'].geometry.iloc[0]
for _, row in sites_utm.iterrows():
    dist_m = s01.distance(row.geometry)
    print(f"  S01 -> {row.site_id} ({row.site_name:<22s}): {dist_m/1000:.1f} km")

print()
# Protected area attributes
print("Protected area properties (Shapely):")
for _, row in protected.iterrows():
    prot_utm = protected.to_crs("EPSG:32620")
    row_utm  = prot_utm[prot_utm.pa_id==row.pa_id].geometry.iloc[0]
    print(f"  {row.pa_id} {row.name:<26s}  area={row_utm.area/1e6:.0f} km²  centroid={row.geometry.centroid.wkt[:30]}")

print()
print("PostGIS equivalents:")
pg = [
    ("ST_Distance(a.geom, b.geom)",              "Distance in CRS units (degrees if EPSG:4326)"),
    ("ST_Distance(ST_Transform(a,32620), ...)",   "Distance in metres (projected)"),
    ("ST_DWithin(a.geom, b.geom, dist)",          "Boolean: within distance (uses spatial index!)"),
    ("ST_Area(geom)",                             "Area in CRS units (m² if projected)"),
    ("ST_Area(ST_Transform(geom, 32620))/1e6",   "Area in km²"),
    ("ST_Centroid(geom)",                         "Centroid of geometry"),
    ("ST_Length(line_geom)",                      "Length of LineString in CRS units"),
    ("ST_Perimeter(poly_geom)",                   "Perimeter of Polygon"),
    ("ST_NumPoints(geom)",                        "Number of vertices"),
    ("ST_GeometryType(geom)",                     "'ST_Point', 'ST_Polygon', etc."),
]
for sql, desc in pg:
    print(f"  {sql:<54s}  {desc}")


---
## Spatial predicates

In [ ]:
print("=== Spatial predicates: ST_Intersects, ST_Within, ST_Contains ===")
print()

# Which monitoring sites fall within each protected area?
print("Sites WITHIN protected areas (ST_Within):")
within = gpd.sjoin(sites, protected, how='inner', predicate='within')
if len(within) > 0:
    for _, row in within.iterrows():
        print(f"  {row.site_id} ({row.site_name_left}) is within {row.name}")
else:
    print("  (No sites fall exactly within the simplified protected area polygons)")
    print("  Demonstrating with buffer approach instead:")

# Buffer sites by 0.3 degrees and check intersection
sites_buf = sites.copy()
sites_buf['geometry'] = sites.geometry.buffer(0.3)
intersecting = gpd.sjoin(sites_buf, protected, how='inner', predicate='intersects')
for _, row in intersecting.iterrows():
    print(f"  {row.site_id} ({row.site_name_left}) intersects {row.name}")

print()
# Which observations are within each watershed?
print("Observations WITHIN each watershed:")
obs_in_ws = gpd.sjoin(observations, watersheds, how='left', predicate='within')
for ws_name, grp in obs_in_ws.groupby('name'):
    species = grp['species'].value_counts().to_dict()
    print(f"  {ws_name}: {dict(species)}")

print()
print("PostGIS spatial predicate functions:")
predicates = [
    ("ST_Intersects(a, b)",  "True if geometries share any space (most common)"),
    ("ST_Within(a, b)",      "True if a is completely inside b"),
    ("ST_Contains(a, b)",    "True if a completely contains b (inverse of Within)"),
    ("ST_Covers(a, b)",      "Like Contains but includes boundary"),
    ("ST_Crosses(a, b)",     "True if geometries cross (lines crossing polygons)"),
    ("ST_Overlaps(a, b)",    "True if geometries overlap and are same dimension"),
    ("ST_Touches(a, b)",     "True if geometries share boundary but not interior"),
    ("ST_Disjoint(a, b)",    "True if geometries share no space (NOT Intersects)"),
    ("ST_DWithin(a, b, d)",  "True if within distance d — uses spatial index"),
    ("ST_Equals(a, b)",      "True if geometrically identical"),
]
print(f"  {'Function':<28s}  Meaning")
print("  " + "-"*65)
for fn, desc in predicates:
    print(f"  {fn:<28s}  {desc}")


---
## Common Pitfalls

**1. ST_Intersects vs ST_Within for point-in-polygon**
`ST_Intersects(point, polygon)` is true for points ON the boundary. `ST_Within(point, polygon)` is true only for points INSIDE (not on boundary). For typical point-in-polygon queries, `ST_Intersects` or `ST_Contains(polygon, point)` is usually correct and uses the spatial index.

**2. ST_Distance not using the spatial index**
`WHERE ST_Distance(a.geom, b.geom) < 1000` computes exact distance for every row pair before filtering — no index. Use `ST_DWithin(a.geom, b.geom, 1000)` which uses the GIST index to pre-filter candidates.

**3. Forgetting to handle invalid geometries**
Real-world data often has self-intersecting polygons, duplicate points, or other invalid geometries. `ST_Intersects` on invalid geometry may return errors or wrong results. Always validate: `WHERE ST_IsValid(geom)` and repair: `ST_MakeValid(geom)` during ingestion.

**4. ST_Buffer on geographic (degree) coordinates**
`ST_Buffer(geom, 1000)` on EPSG:4326 geometry creates a buffer of 1000 DEGREES (not metres). Project first: `ST_Buffer(ST_Transform(geom, 32620), 1000)` for a 1km buffer.

**5. PostGIS not installed or not enabled on the database**
`CREATE EXTENSION postgis;` must be run once per database (requires superuser). A common error is connecting to a database where PostGIS is installed on the server but the extension has not been enabled in that specific database.

**6. Using ST_Equals for approximate geometry comparison**
`ST_Equals` requires geometrically identical geometries (same vertices in same order). For approximate equality (same location within a tolerance), use `ST_DWithin(a, b, 0.0001)` or `ST_Equals(ST_SnapToGrid(a, 0.001), ST_SnapToGrid(b, 0.001))`.


---
*sql_methods_library - Samantha McGarrigle*